In [1]:
import numpy as np
import pandas as pd

input_file = "train.csv"
df_train = pd.read_csv(input_file)
df_train.shape

(7352, 563)

In [2]:
df_train["subject"].value_counts()

subject
25    409
21    408
26    392
30    383
28    382
27    376
23    372
17    368
16    366
19    360
1     347
29    344
3     341
15    328
6     325
14    323
22    321
11    316
7     308
5     302
8     281
Name: count, dtype: int64

In [3]:
df_train["Activity"].value_counts()

Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64

In [4]:
from sklearn.preprocessing import OrdinalEncoder

df_train_cat = df_train[["Activity"]]
ordinal_encoder = OrdinalEncoder()
y_train = ordinal_encoder.fit_transform(df_train_cat).flatten()
ordinal_encoder.categories_

[array(['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS',
        'WALKING_UPSTAIRS'], dtype=object)]

In [5]:
df_train_num = df_train.drop(["subject", "Activity"], axis=1)
subject = df_train["subject"]

# 특성 엔지니어링 1: 사람별 평균 빼기 (사람마다 다른 폰 착용 각도·걸음 습관 제거, subject 는 묶음 기준으로만 사용)
df_center = df_train_num - df_train_num.groupby(subject).transform("mean")

# 특성 엔지니어링 2: 같은 사람의 앞뒤 측정 구간 평균 (짧은 창 5, 긴 창 75)
def rolling_mean(df, w):
    return df.groupby(subject).transform(lambda s: s.rolling(w, center=True, min_periods=1).mean())

X_train = np.hstack([df_center.to_numpy(), rolling_mean(df_center, 5).to_numpy(), rolling_mean(df_center, 75).to_numpy()])
X_raw = df_train_num.to_numpy()
X_train.shape

(7352, 1683)

In [6]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [7]:
# 1) 모델 비교: 원본 특성 vs 내 특성, 여러 분류 모델 (10-fold)
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "SGD": SGDClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=3000),
    "LinearSVC": LinearSVC(max_iter=10000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "SVC": SVC(C=10),
}
for name, model in models.items():
    raw = cross_val_score(model, X_raw, y_train, cv=kfold, scoring="accuracy").mean()
    fe = cross_val_score(model, X_train, y_train, cv=kfold, scoring="accuracy").mean()
    print(f"{name:20s} 원본 {raw:.4f}   내 특성 {fe:.4f}")

SGD                  원본 0.9771   내 특성 0.9966


LogisticRegression   원본 0.9848   내 특성 0.9993


LinearSVC            원본 0.9853   내 특성 0.9995


DecisionTree         원본 0.9441   내 특성 0.9879


RandomForest         원본 0.9822   내 특성 0.9977


SVC                  원본 0.9876   내 특성 0.9999


In [8]:
# 2) 하이퍼파라미터 탐색: 가장 좋은 SVC 의 C, gamma (Grid Search, 5-fold)
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [1, 10, 100], "gamma": ["scale", 0.0002, 0.001]}
grid_search = GridSearchCV(SVC(), param_grid, cv=KFold(n_splits=5, shuffle=True, random_state=42),
                           scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_, grid_search.best_score_)
for mean, params in zip(grid_search.cv_results_["mean_test_score"], grid_search.cv_results_["params"]):
    print(f"{mean:.4f}", params)

{'C': 10, 'gamma': 'scale'} 0.9998639455782312
0.9985 {'C': 1, 'gamma': 'scale'}
0.9686 {'C': 1, 'gamma': 0.0002}
0.9937 {'C': 1, 'gamma': 0.001}
0.9999 {'C': 10, 'gamma': 'scale'}
0.9954 {'C': 10, 'gamma': 0.0002}
0.9985 {'C': 10, 'gamma': 0.001}
0.9999 {'C': 100, 'gamma': 'scale'}
0.9993 {'C': 100, 'gamma': 0.0002}
0.9996 {'C': 100, 'gamma': 0.001}


In [9]:
# 3) 최종 모델 10-fold 교차 검증
final_clf = grid_search.best_estimator_

cv_res = cross_val_score(final_clf, X_train, y_train, cv=kfold, scoring="accuracy")
cv_res

array([1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 0.99863946])

In [10]:
np.mean(cv_res)

np.float64(0.9998639455782312)

In [11]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

y_pred = cross_val_predict(final_clf, X_train, y_train, cv=kfold)
confusion_matrix(y_train, y_pred)

array([[1407,    0,    0,    0,    0,    0],
       [   0, 1286,    0,    0,    0,    0],
       [   0,    1, 1373,    0,    0,    0],
       [   0,    0,    0, 1226,    0,    0],
       [   0,    0,    0,    0,  986,    0],
       [   0,    0,    0,    0,    0, 1073]])